In [ ]:
# 프로젝트 위치가 GummyBearing인지 확인

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()

if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError("프로젝트의 src 폴더를 찾을 수 없습니다.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.xjtu_features import extract_xjtu_features

In [ ]:
# 함수를 불러오고 데이터 경로와 베어링 폴더 확인

import numpy as np
import pandas as pd
from IPython.display import display

from src.xjtu_features import (
    extract_xjtu_features, find_bearing_folders,
    BEARING_METADATA, FEATURE_COLUMNS, VALUE_COLUMNS, ERROR_COLUMNS,
)

# 압축을 푼 세 조건의 폴더가 들어 있는 상위 경로로 수정한다.
DATA_ROOT = PROJECT_ROOT / "DATA" / "XJTU_SY_Bearing_Dataset"

# 기본은 15개 전체. 일부만 시험하려면 ["Bearing1_1"] 등으로 바꾼다.
selected_bearings = list(BEARING_METADATA)
bearing_folders = find_bearing_folders(DATA_ROOT, selected_bearings)

for bearing, folder in bearing_folders.items():
    count = sum(p.is_file() and p.suffix.lower() == ".csv" for p in folder.iterdir())
    expected = BEARING_METADATA[bearing]["expected_files"]
    print(f"{bearing}: CSV {count}개 / PDF 기준 {expected}개")

# timestamp: 파일명 숫자(1, 2, ...), 실제 날짜가 아닌 1분 간격 측정 순서.
# fault_element: 종료 후 확인된 결함 부위. 개별 시점의 정상/고장 라벨이 아니다.
# 메타데이터 출처: Introduction_to_XJTU-SY_Bearing_Dataset, p.3 Table 2.

Bearing1_1: CSV 123개 / PDF 기준 123개
Bearing1_2: CSV 161개 / PDF 기준 161개
Bearing1_3: CSV 158개 / PDF 기준 158개
Bearing1_4: CSV 122개 / PDF 기준 122개
Bearing1_5: CSV 52개 / PDF 기준 52개
Bearing2_1: CSV 491개 / PDF 기준 491개
Bearing2_2: CSV 161개 / PDF 기준 161개
Bearing2_3: CSV 533개 / PDF 기준 533개
Bearing2_4: CSV 42개 / PDF 기준 42개
Bearing2_5: CSV 339개 / PDF 기준 339개
Bearing3_1: CSV 2538개 / PDF 기준 2538개
Bearing3_2: CSV 2496개 / PDF 기준 2496개
Bearing3_3: CSV 371개 / PDF 기준 371개
Bearing3_4: CSV 1515개 / PDF 기준 1515개
Bearing3_5: CSV 114개 / PDF 기준 114개


In [ ]:
# 특징 계산 실행 (원본 CSV 하나 → 수평·수직 특징을 담은 한 행)

results = {}
error_results = {}

for bearing, folder in bearing_folders.items():
    features_df, errors_df = extract_xjtu_features(
        folder_path=folder,
        bearing=bearing,
    )
    results[bearing] = features_df
    error_results[bearing] = errors_df
    print(f"{bearing}: 결과 {features_df.shape}, 오류 항목 {len(errors_df)}개")

Bearing1_1: 결과 (123, 14), 오류 항목 0개
Bearing1_2: 결과 (161, 14), 오류 항목 0개
Bearing1_3: 결과 (158, 14), 오류 항목 0개
Bearing1_4: 결과 (122, 14), 오류 항목 0개
Bearing1_5: 결과 (52, 14), 오류 항목 0개
Bearing2_1: 결과 (491, 14), 오류 항목 0개
Bearing2_2: 결과 (161, 14), 오류 항목 0개
Bearing2_3: 결과 (533, 14), 오류 항목 0개
Bearing2_4: 결과 (42, 14), 오류 항목 0개
Bearing2_5: 결과 (339, 14), 오류 항목 0개
Bearing3_1: 결과 (2538, 14), 오류 항목 0개
Bearing3_2: 결과 (2496, 14), 오류 항목 0개
Bearing3_3: 결과 (371, 14), 오류 항목 0개
Bearing3_4: 결과 (1515, 14), 오류 항목 0개
Bearing3_5: 결과 (114, 14), 오류 항목 0개


In [ ]:
# 결과 확인

summary_df = pd.DataFrame([
    {
        "bearing": bearing,
        "condition": BEARING_METADATA[bearing]["condition"],
        "expected_rows": BEARING_METADATA[bearing]["expected_files"],
        "result_rows": len(features_df),
        "error_count": len(error_results[bearing]),
    }
    for bearing, features_df in results.items()
])
display(summary_df)

xjtu_df = pd.concat(results.values(), ignore_index=True)
xjtu_df = xjtu_df.sort_values(["condition", "bearing", "timestamp"]).reset_index(drop=True)

# 14열을 생략하지 않고 미리 본다. 저장할 때는 반올림하지 않는다.
with pd.option_context("display.max_columns", None, "display.precision", 4):
    display(xjtu_df.head())
print("통합 결과 크기:", xjtu_df.shape)

,bearing,condition,expected_rows,result_rows,error_count
0,Bearing1_1,1,123,123,0
1,Bearing1_2,1,161,161,0
2,Bearing1_3,1,158,158,0
3,Bearing1_4,1,122,122,0
4,Bearing1_5,1,52,52,0
5,Bearing2_1,2,491,491,0
6,Bearing2_2,2,161,161,0
7,Bearing2_3,2,533,533,0
8,Bearing2_4,2,42,42,0
9,Bearing2_5,2,339,339,0


,timestamp,bearing,condition,rpm,load_kn,fault_element,horizontal_rms,horizontal_kurtosis,horizontal_peak,horizontal_crest_factor,vertical_rms,vertical_kurtosis,vertical_peak,vertical_crest_factor
0,1,Bearing1_1,1,2100,12,Outer race,0.5639,3.0714,2.5299,4.4865,0.5605,2.9753,2.4693,4.4058
1,2,Bearing1_1,1,2100,12,Outer race,0.5891,3.1379,3.6231,6.1505,0.5669,2.9642,2.8123,4.9613
2,3,Bearing1_1,1,2100,12,Outer race,0.5895,3.2463,3.3218,5.6346,0.5689,2.9650,2.3274,4.0913
3,4,Bearing1_1,1,2100,12,Outer race,0.5973,3.2482,2.8726,4.8096,0.5688,2.9236,2.6929,4.7345
4,5,Bearing1_1,1,2100,12,Outer race,0.6046,3.3939,4.1369,6.8419,0.5830,3.0356,2.6558,4.5558


통합 결과 크기: (9216, 14)


In [ ]:
# 오류와 계산된 특징 확인 (원본 결측치 일괄 검사는 팀원 담당)

all_errors_df = pd.concat(error_results.values(), ignore_index=True).reindex(columns=ERROR_COLUMNS)
print("오류 항목 수:", len(all_errors_df))
if not all_errors_df.empty:
    display(all_errors_df)

print("계산된 특징의 NaN 개수:")
display(xjtu_df[VALUE_COLUMNS].isna().sum())

# 다른 베어링에서 timestamp가 같은 것은 정상이다.
duplicate_mask = xjtu_df.duplicated(["bearing", "timestamp"], keep=False)
print("중복된 bearing + timestamp 행 수:", int(duplicate_mask.sum()))
if duplicate_mask.any():
    display(xjtu_df.loc[duplicate_mask])

print("숫자 정렬 및 파일 번호 누락·중복 확인은 추출 함수에서 수행합니다.")
print("전처리에서는 보간·이상치 삭제·정규화·이동평균을 적용하지 않습니다.")

오류 항목 수: 0
계산된 특징의 NaN 개수:


horizontal_rms             0
horizontal_kurtosis        0
horizontal_peak            0
horizontal_crest_factor    0
vertical_rms               0
vertical_kurtosis          0
vertical_peak              0
vertical_crest_factor      0
dtype: int64

중복된 bearing + timestamp 행 수: 0
숫자 정렬 및 파일 번호 누락·중복 확인은 추출 함수에서 수행합니다.
전처리에서는 보간·이상치 삭제·정규화·이동평균을 적용하지 않습니다.


In [ ]:
# 베어링별 CSV와 통합 CSV 저장

OUTPUT_ROOT = PROJECT_ROOT / "DATA" / "features"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ERROR_ROOT = OUTPUT_ROOT / "errors"
ERROR_ROOT.mkdir(parents=True, exist_ok=True)

# 저장 직전 현재 결과에서 통합본과 오류 목록을 다시 구성한다.
xjtu_df = pd.concat(results.values(), ignore_index=True).reindex(columns=FEATURE_COLUMNS)
xjtu_df = xjtu_df.sort_values(["condition", "bearing", "timestamp"]).reset_index(drop=True)
all_errors_df = pd.concat(error_results.values(), ignore_index=True).reindex(columns=ERROR_COLUMNS)
all_errors_df.to_csv(ERROR_ROOT / "XJTU_SY_errors.csv", index=False, encoding="utf-8-sig")

has_invalid_features = not np.isfinite(xjtu_df[VALUE_COLUMNS].to_numpy(dtype=float)).all()
has_duplicates = xjtu_df.duplicated(["bearing", "timestamp"]).any()
has_wrong_count = any(
    len(frame) != BEARING_METADATA[bearing]["expected_files"]
    for bearing, frame in results.items()
)
if xjtu_df.empty or not all_errors_df.empty or has_invalid_features or has_duplicates or has_wrong_count:
    raise ValueError("오류/특징값/파일 개수를 확인하세요. 오류 목록만 저장했고 특징 CSV는 갱신하지 않았습니다.")

for bearing, features_df in results.items():
    # Bearing1_1 → XJTU_SY_bearing_1_1_features.csv
    bearing_id = bearing.removeprefix("Bearing")
    filename = f"XJTU_SY_bearing_{bearing_id}_features.csv"
    features_df.to_csv(OUTPUT_ROOT / filename, index=False, encoding="utf-8-sig")

# 일부 베어링으로 시험한 결과를 전체 통합본으로 저장하지 않는다.
is_full_dataset = set(results) == set(BEARING_METADATA)
total_name = "XJTU_SY_total_features.csv" if is_full_dataset else "XJTU_SY_selected_features.csv"
xjtu_df.to_csv(OUTPUT_ROOT / total_name, index=False, encoding="utf-8-sig")
print(f"베어링별 {len(results)}개 + 통합 1개 저장 완료!!")
print("통합 파일:", total_name, "| 크기:", xjtu_df.shape)

베어링별 15개 + 통합 1개 저장 완료!!
통합 파일: XJTU_SY_total_features.csv | 크기: (9216, 14)
